# Check what is the HPC lab / port

In [2]:
import socket
import os
import subprocess
import re

# 1. Get HPC node name
hostname = socket.gethostname()
print("=" * 60)
print("HPC NODE INFORMATION")
print("=" * 60)
print(f"Node name: {hostname}")
print()

HPC NODE INFORMATION
Node name: gpu8.hpc.pub.lan



In [ ]:
# 2. GPU Information
print("=" * 60)
print("GPU INFORMATION")
print("=" * 60)
try:
    import torch
    if torch.cuda.is_available():
        print(f"CUDA Available: Yes")
        print(f"Number of GPUs: {torch.cuda.device_count()}")
        for i in range(torch.cuda.device_count()):
            print(f"\nGPU {i}:")
            print(f"  Name: {torch.cuda.get_device_name(i)}")
            print(f"  Compute Capability: {torch.cuda.get_device_capability(i)}")
            print(f"  Total Memory: {torch.cuda.get_device_properties(i).total_memory / 1024**3:.2f} GB")
        print(f"\nCurrent CUDA Device: {torch.cuda.current_device()}")
        print(f"CUDA Version: {torch.version.cuda}")
    else:
        print("CUDA Available: No")
except ImportError:
    print("PyTorch not installed")

# Try nvidia-smi for detailed GPU info
print("\n" + "=" * 60)
print("NVIDIA-SMI Output")
print("=" * 60)
try:
    result = subprocess.run(['nvidia-smi'], capture_output=True, text=True, timeout=5)
    if result.returncode == 0:
        print(result.stdout)
    else:
        print("nvidia-smi command failed")
except Exception as e:
    print(f"nvidia-smi not available: {e}")

# 3. CPU Information
print("\n" + "=" * 60)
print("CPU INFORMATION")
print("=" * 60)
try:
    # Read CPU info
    with open('/proc/cpuinfo', 'r') as f:
        cpuinfo = f.read()
    
    # Extract model name
    model_match = re.search(r'model name\s*:\s*(.+)', cpuinfo)
    if model_match:
        print(f"CPU Model: {model_match.group(1)}")
    
    # Count cores
    cpu_count = cpuinfo.count('processor')
    print(f"Number of CPU cores: {cpu_count}")
    
    # Physical cores
    physical_cores = cpuinfo.count('core id')
    if physical_cores > 0:
        unique_cores = len(set(re.findall(r'core id\s*:\s*(\d+)', cpuinfo)))
        print(f"Physical CPU cores: {unique_cores}")
    
except Exception as e:
    print(f"Error reading CPU info: {e}")

# 4. Memory Information
print("\n" + "=" * 60)
print("MEMORY INFORMATION")
print("=" * 60)
try:
    with open('/proc/meminfo', 'r') as f:
        meminfo = f.read()
    
    mem_total = re.search(r'MemTotal:\s*(\d+)', meminfo)
    mem_available = re.search(r'MemAvailable:\s*(\d+)', meminfo)
    
    if mem_total:
        total_gb = int(mem_total.group(1)) / 1024 / 1024
        print(f"Total RAM: {total_gb:.2f} GB")
    
    if mem_available:
        available_gb = int(mem_available.group(1)) / 1024 / 1024
        print(f"Available RAM: {available_gb:.2f} GB")
        
except Exception as e:
    print(f"Error reading memory info: {e}")

# 5. OS Information
print("\n" + "=" * 60)
print("OPERATING SYSTEM INFORMATION")
print("=" * 60)
try:
    with open('/etc/os-release', 'r') as f:
        os_info = f.read()
    
    name = re.search(r'PRETTY_NAME="(.+)"', os_info)
    if name:
        print(f"OS: {name.group(1)}")
    
    print(f"Kernel: {os.uname().release}")
    
except Exception as e:
    print(f"Error reading OS info: {e}")

print("\n" + "=" * 60)